# GTSRB - German Traffic Sign Recognition BenchmarkThis notebook downloads the dataset, explores it, and preprocesses images.**Dataset:** https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/data

In [ ]:
# Install kagglehub if not already installed
!pip install kagglehub


In [ ]:
import kagglehub
import os
import shutil

# Target directory (next to this notebook)
target_dir = os.path.join(os.getcwd(), "dataset")

# Download the dataset from Kaggle
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
print(f"Kaggle downloaded to: {path}")

# Copy to project directory if not already there
if os.path.abspath(path) != os.path.abspath(target_dir):
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    shutil.copytree(path, target_dir)
    path = target_dir

print(f"Dataset available at: {path}")
print(f"Contents: {os.listdir(path)}")


In [ ]:
# List the contents of the downloaded dataset
!ls -la {path}


## Basic Data Exploration

In [ ]:
import os

# List the dataset folder structure
for item in sorted(os.listdir(path)):
    item_path = os.path.join(path, item)
    if os.path.isdir(item_path):
        count = len(os.listdir(item_path))
        print(f"  📁 {item}/  ({count} items)")
    else:
        print(f"  📄 {item}")


In [ ]:
import pandas as pd

# Load the training labels CSV (lives at dataset root)
df = pd.read_csv(os.path.join(path, "Train.csv"))
print(f"Training samples: {len(df)}")
print()
df.head(10)


In [ ]:
# Class distribution
print(f"Number of classes: {df['ClassId'].nunique()}")
print()
counts = df['ClassId'].value_counts().sort_index()
counts


In [ ]:
import matplotlib.pyplot as plt

# Plot class distribution
counts = df['ClassId'].value_counts().sort_index()
plt.figure(figsize=(14, 5))
plt.bar(counts.index, counts.values)
plt.xlabel("Class ID")
plt.ylabel("Number of Samples")
plt.title("Training Set Class Distribution")
plt.xticks(range(0, 43))
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display sample images from each class
fig, axes = plt.subplots(5, 9, figsize=(20, 12))
axes = axes.flatten()

for cls in sorted(df['ClassId'].unique()):
    sample = df[df["ClassId"] == cls].iloc[0]
    # Path column already contains relative path from dataset root
    img_path = os.path.join(path, sample['Path'])
    img = Image.open(img_path)
    axes[cls].imshow(img)
    axes[cls].set_title(f"Class {cls}\n{img.size[1]}x{img.size[0]}")
    axes[cls].axis("off")

plt.suptitle("One Sample from Each Class", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Image size statistics
heights = df['Height'].value_counts().sort_index()
widths = df['Width'].value_counts().sort_index()

print("Image heights:")
print(heights)
print()
print("Image widths:")
print(widths)


In [ ]:
# Check test set
test_dir = os.path.join(path, "Test")
test_images = os.listdir(test_dir)
print(f"Test images: {len(test_images)}")

# Load test labels
df_test = pd.read_csv(os.path.join(path, "Test.csv"))
print(f"Test labels: {len(df_test)}")
print()
df_test.head()


## Image Preprocessing & Data Augmentation

In [ ]:
import numpy as np
from PIL import Image

IMG_SIZE = (32, 32)


def load_image(img_path):
    """Load an image, resize to IMG_SIZE, normalize pixels to [0, 1]."""
    img = Image.open(img_path).convert("RGB")
    img = img.resize(IMG_SIZE)
    return np.array(img) / 255.0


# Quick test on one image
test_img = load_image(os.path.join(path, df.iloc[0]["Path"]))
print(f"Shape: {test_img.shape}, dtype: {test_img.dtype}")
print(f"Pixel range: [{test_img.min():.3f}, {test_img.max():.3f}]")


In [ ]:
import tqdm

# Build image array and labels
X_train = np.zeros((len(df), IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)
y_train = df['ClassId'].values.astype(np.int32)

for i, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Loading training images"):
    img_path = os.path.join(path, row['Path'])
    X_train[i] = load_image(img_path)

print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


In [ ]:
from sklearn.model_selection import train_test_split

# Stratified 80/20 split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print(f"Train: {X_tr.shape[0]} samples, Val: {X_val.shape[0]} samples")

# Verify class balance in both splits
print(f"\nTrain class distribution:\n{pd.Series(y_tr).value_counts().sort_index()}")
print(f"\nVal class distribution:\n{pd.Series(y_val).value_counts().sort_index()}")


In [ ]:
# Compute class weights (inverse frequency)
num_classes = len(np.unique(y_tr))
class_counts = np.bincount(y_tr, minlength=num_classes)
class_weights = len(y_tr) / (num_classes * class_counts)

# Plot class weights
plt.figure(figsize=(12, 4))
plt.bar(range(num_classes), class_weights)
plt.xlabel("Class ID")
plt.ylabel("Weight")
plt.title("Class Weights (Inverse Frequency)")
plt.xticks(range(0, 43))
plt.tight_layout()
plt.show()


In [ ]:
import tensorflow as tf

# Define data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(factor=0.15, fill_mode="nearest"),
    tf.keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1, fill_mode="nearest"),
    tf.keras.layers.RandomContrast(factor=0.2),
], name="augmentation")

# Note: No horizontal flip -- direction/text on signs matters!



In [ ]:
# Show original + several augmented versions of one image
sample = X_tr[0:1]  # keep batch dim

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample[0])
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1, 5):
    augmented = data_augmentation(sample)
    axes[i].imshow(augmented[0])
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
BATCH_SIZE = 64

# Create tf.data datasets with augmentation applied only to training data
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
    .shuffle(1000)
    .map(lambda x, y: (data_augmentation(x), y))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"Train batches: {train_ds.cardinality.numpy()}, Val batches: {val_ds.cardinality.numpy()}")

# Inspect one batch
for x_batch, y_batch in train_ds.take(1):
    print(f"Batch shape: {x_batch.shape}, Labels shape: {y_batch.shape}")
